In [1]:
# =========================
# DANN + FixMatch (start from finetune checkpoint)
#   Train with:
#       - PlantDoc/train_labeled
#       - PlantDoc/train_unlabeled_weak
#       - PlantDoc/train_unlabeled_strong
#       - PlantVillage (for domain loss only)
#   Evaluate on PlantDoc/test
# =========================

import os
import time
import json
import random
import shutil
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================
# Paths and config
# =========================

DRIVE_ROOT = "/content/drive/MyDrive/dataset"

# Choose ONE starting checkpoint
START_CKPT = "/content/drive/MyDrive/plant_disease_project/runs_finetune_supervised_new/finetune_best.pt"
#START_CKPT = "/content/drive/MyDrive/plant_disease_project/runs_supervised_baseline/best.pt"

DRIVE_SOURCE_DIR = f"{DRIVE_ROOT}/PlantVillage"
DRIVE_PD_LABELED = f"{DRIVE_ROOT}/PlantDoc/train_labeled"
DRIVE_PD_UNLABELED_WEAK = f"{DRIVE_ROOT}/PlantDoc/train_unlabeled_weak"
DRIVE_PD_UNLABELED_STRONG = f"{DRIVE_ROOT}/PlantDoc/train_unlabeled_strong"
DRIVE_PD_TEST = f"{DRIVE_ROOT}/PlantDoc/test"

OUT_DIR = "/content/drive/MyDrive/plant_disease_project/runs_dann_fixmatch_v2"
os.makedirs(OUT_DIR, exist_ok=True)

COPY_TO_LOCAL = True

MODEL_NAME = "resnet50"   # must match checkpoint model
EPOCHS = 20
WARMUP_EPOCHS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE_SOURCE = 64
BATCH_SIZE_LABELED = 64
BATCH_SIZE_UNLABELED = 64
BATCH_SIZE_TEST = 64
NUM_WORKERS = 2
USE_AMP = True
SEED = 42

THRESHOLD = 0.90
LAMBDA_U = 0.25
LAMBDA_D_MAX = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Info] device =", device)

[Info] device = cuda


In [3]:
# =========================
# Checks
# =========================
if not os.path.isfile(START_CKPT):
    raise RuntimeError(f"[ERROR] Starting checkpoint not found: {START_CKPT}")

for p in [DRIVE_SOURCE_DIR, DRIVE_PD_LABELED, DRIVE_PD_UNLABELED_WEAK, DRIVE_PD_UNLABELED_STRONG, DRIVE_PD_TEST]:
    if not os.path.isdir(p):
        raise RuntimeError(f"[ERROR] Folder not found: {p}")

empty_dirs = []
for c in os.listdir(DRIVE_PD_LABELED):
    p = os.path.join(DRIVE_PD_LABELED, c)
    if os.path.isdir(p) and len(os.listdir(p)) == 0:
        empty_dirs.append(p)

for p in empty_dirs:
    print("[Removing empty folder]", p)
    shutil.rmtree(p)

print(f"[Info] Removed {len(empty_dirs)} empty folders from train_labeled")

[Info] Removed 0 empty folders from train_labeled


In [4]:
# =========================
# Load checkpoint metadata
# =========================
ckpt = torch.load(START_CKPT, map_location="cpu")
print("[Info] checkpoint keys:", ckpt.keys())

if "classes" not in ckpt:
    raise RuntimeError("Checkpoint does not contain 'classes'.")

source_classes = ckpt["classes"]
num_classes = len(source_classes)
source_class_to_idx = {name: i for i, name in enumerate(source_classes)}

print("[Info] num_classes =", num_classes)
print("[Info] source classes =", source_classes)

[Info] checkpoint keys: dict_keys(['model_state_dict', 'best_acc', 'epoch', 'classes', 'model_name', 'baseline_ckpt'])
[Info] num_classes = 15
[Info] source classes = ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [5]:
# =========================
# Transforms
# =========================
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])

source_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    normalize,
])

labeled_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    normalize,
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

ssl_file_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

In [6]:
# =========================
# Class mapping
# =========================
PV2PD = {
    "Pepper__bell___Bacterial_spot": "Bell_pepper leaf spot",
    "Pepper__bell___healthy": "Bell_pepper leaf",

    "Potato___Early_blight": "Potato leaf early blight",
    "Potato___Late_blight": "Potato leaf late blight",
    "Potato___healthy": None,

    "Tomato_Bacterial_spot": "Tomato leaf bacterial spot",
    "Tomato_Early_blight": "Tomato Early blight leaf",
    "Tomato_Late_blight": "Tomato leaf late blight",
    "Tomato_Leaf_Mold": "Tomato mold leaf",
    "Tomato_Septoria_leaf_spot": "Tomato Septoria leaf spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato two spotted spider mites leaf",
    "Tomato__Target_Spot": None,
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato leaf yellow virus",
    "Tomato__Tomato_mosaic_virus": "Tomato leaf mosaic virus",
    "Tomato_healthy": "Tomato leaf",
}

pd_folder_to_src_idx = {}
for pv_name, pd_name in PV2PD.items():
    if pd_name is None:
        continue
    if pv_name not in source_class_to_idx:
        continue
    pd_folder_to_src_idx[pd_name] = source_class_to_idx[pv_name]

mapped_pd_classes = sorted(pd_folder_to_src_idx.keys())

print("[Info] PlantDoc folder -> source label index mapping")
for k, v in pd_folder_to_src_idx.items():
    print(f"  {k} -> {v}")

[Info] PlantDoc folder -> source label index mapping
  Bell_pepper leaf spot -> 0
  Bell_pepper leaf -> 1
  Potato leaf early blight -> 2
  Potato leaf late blight -> 3
  Tomato leaf bacterial spot -> 5
  Tomato Early blight leaf -> 6
  Tomato leaf late blight -> 7
  Tomato mold leaf -> 8
  Tomato Septoria leaf spot -> 9
  Tomato two spotted spider mites leaf -> 10
  Tomato leaf yellow virus -> 12
  Tomato leaf mosaic virus -> 13
  Tomato leaf -> 14


In [7]:
# =========================
# Dataset wrappers
# =========================
class MappedImageFolderDataset(Dataset):
    def __init__(self, imagefolder_ds, folder_to_newlabel):
        self.base = imagefolder_ds
        self.folder_to_newlabel = folder_to_newlabel
        self.keep_indices = []

        for i, (_, y) in enumerate(self.base.samples):
            folder_name = self.base.classes[y]
            if folder_name in self.folder_to_newlabel:
                self.keep_indices.append(i)

        self.class_counts = {}
        for i in self.keep_indices:
            _, y = self.base.samples[i]
            folder_name = self.base.classes[y]
            self.class_counts[folder_name] = self.class_counts.get(folder_name, 0) + 1

    def __len__(self):
        return len(self.keep_indices)

    def __getitem__(self, idx):
        real_idx = self.keep_indices[idx]
        x, old_y = self.base[real_idx]
        folder_name = self.base.classes[old_y]
        new_y = self.folder_to_newlabel[folder_name]
        return x, new_y


class PairedMappedUnlabeledDataset(Dataset):
    def __init__(self, weak_root, strong_root, allowed_classes, transform=None):
        self.weak_root = weak_root
        self.strong_root = strong_root
        self.allowed_classes = set(allowed_classes)
        self.transform = transform
        self.samples = []
        self.skipped_unmapped_classes = []

        weak_classes = sorted([d for d in os.listdir(weak_root) if os.path.isdir(os.path.join(weak_root, d))])
        for cls_name in weak_classes:
            if cls_name not in self.allowed_classes:
                self.skipped_unmapped_classes.append(cls_name)
                continue
            weak_cls_dir = os.path.join(weak_root, cls_name)
            strong_cls_dir = os.path.join(strong_root, cls_name)
            if not os.path.isdir(strong_cls_dir):
                continue

            weak_files = sorted([f for f in os.listdir(weak_cls_dir) if os.path.isfile(os.path.join(weak_cls_dir, f))])
            strong_file_set = set([f for f in os.listdir(strong_cls_dir) if os.path.isfile(os.path.join(strong_cls_dir, f))])

            for fn in weak_files:
                if fn in strong_file_set:
                    weak_path = os.path.join(weak_cls_dir, fn)
                    strong_path = os.path.join(strong_cls_dir, fn)
                    self.samples.append((weak_path, strong_path, cls_name))

        self.class_counts = {}
        for _, _, cls_name in self.samples:
            self.class_counts[cls_name] = self.class_counts.get(cls_name, 0) + 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        from PIL import Image
        weak_path, strong_path, _ = self.samples[idx]
        weak_img = Image.open(weak_path).convert("RGB")
        strong_img = Image.open(strong_path).convert("RGB")
        if self.transform is not None:
            weak_img = self.transform(weak_img)
            strong_img = self.transform(strong_img)
        return weak_img, strong_img

In [8]:
# =========================
# Load datasets
# =========================
src_ds = datasets.ImageFolder(DRIVE_SOURCE_DIR, transform=source_tf)
pd_labeled_raw = datasets.ImageFolder(DRIVE_PD_LABELED, transform=labeled_tf)
pd_test_raw = datasets.ImageFolder(DRIVE_PD_TEST, transform=eval_tf)

pd_labeled_ds = MappedImageFolderDataset(pd_labeled_raw, pd_folder_to_src_idx)
pd_test_ds = MappedImageFolderDataset(pd_test_raw, pd_folder_to_src_idx)
pd_unlabeled_ds = PairedMappedUnlabeledDataset(
    DRIVE_PD_UNLABELED_WEAK, DRIVE_PD_UNLABELED_STRONG, mapped_pd_classes, ssl_file_tf
)

print("[Info] raw source classes:", src_ds.classes)
print("[Info] raw train_labeled classes:", pd_labeled_raw.classes)
print("[Info] raw test classes:", pd_test_raw.classes)
print("[Info] mapped train_labeled size:", len(pd_labeled_ds))
print("[Info] mapped test size:", len(pd_test_ds))
print("[Info] mapped unlabeled pairs:", len(pd_unlabeled_ds))
print("[Info] unlabeled skipped unmapped classes:", pd_unlabeled_ds.skipped_unmapped_classes)

if len(pd_labeled_ds) == 0 or len(pd_test_ds) == 0 or len(pd_unlabeled_ds) == 0:
    raise RuntimeError("One of the mapped datasets is empty.")

[Info] raw source classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
[Info] raw train_labeled classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', '

In [9]:
# =========================
# DataLoaders
# =========================
src_loader = DataLoader(src_ds, batch_size=BATCH_SIZE_SOURCE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=(device.type=="cuda"), drop_last=False)
pd_labeled_loader = DataLoader(pd_labeled_ds, batch_size=BATCH_SIZE_LABELED, shuffle=True, num_workers=NUM_WORKERS,
                               pin_memory=(device.type=="cuda"), drop_last=False)
pd_unlabeled_loader = DataLoader(pd_unlabeled_ds, batch_size=BATCH_SIZE_UNLABELED, shuffle=True, num_workers=NUM_WORKERS,
                                 pin_memory=(device.type=="cuda"), drop_last=False)
pd_test_loader = DataLoader(pd_test_ds, batch_size=BATCH_SIZE_TEST, shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=(device.type=="cuda"), drop_last=False)

print("[Info] source batches:", len(src_loader))
print("[Info] labeled batches:", len(pd_labeled_loader))
print("[Info] unlabeled batches:", len(pd_unlabeled_loader))
print("[Info] test batches:", len(pd_test_loader))

[Info] source batches: 323
[Info] labeled batches: 2
[Info] unlabeled batches: 15
[Info] test batches: 2


In [10]:
# =========================
# Correct model loading
# =========================
def build_plain_resnet(name: str, num_classes: int):
    name = name.lower()
    if name == "resnet18":
        m = models.resnet18(weights=None)
    elif name == "resnet50":
        m = models.resnet50(weights=None)
    else:
        raise ValueError("Only resnet18/resnet50 supported.")
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

plain_model = build_plain_resnet(MODEL_NAME, num_classes)

if "model_state_dict" in ckpt:
    state = ckpt["model_state_dict"]
elif "model" in ckpt:
    state = ckpt["model"]
else:
    raise RuntimeError("Cannot find model weights inside checkpoint.")

load_info = plain_model.load_state_dict(state, strict=False)
print("[Info] plain_model missing keys:", load_info.missing_keys)
print("[Info] plain_model unexpected keys:", load_info.unexpected_keys)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)

class DANNResNet(nn.Module):
    def __init__(self, plain_resnet, num_classes):
        super().__init__()
        self.backbone = plain_resnet
        feat_dim = self.backbone.fc.in_features

        old_fc = self.backbone.fc
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Linear(feat_dim, num_classes)
        self.classifier.load_state_dict(old_fc.state_dict())

        self.domain_classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def extract_features(self, x):
        return self.backbone(x)

    def classify(self, f):
        return self.classifier(f)

    def domain_logits(self, f, lambd=1.0):
        return self.domain_classifier(grad_reverse(f, lambd)).squeeze(1)

    def forward(self, x):
        f = self.extract_features(x)
        return self.classify(f)

model = DANNResNet(plain_model, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
dom_criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type=="cuda"))

print("[Info] Correct checkpoint loading complete.")

[Info] plain_model missing keys: []
[Info] plain_model unexpected keys: []
[Info] Correct checkpoint loading complete.


In [11]:
# =========================
# Eval helpers
# =========================
@torch.no_grad()
def confusion_matrix(pred: List[int], true: List[int], num_classes: int):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for p, t in zip(pred, true):
        cm[t, p] += 1
    return cm

def macro_f1_from_cm(cm):
    f1s = []
    for c in range(cm.size(0)):
        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        f1s.append(f1)
    return float(sum(f1s) / len(f1s)) if f1s else 0.0

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_true = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        pred = logits.argmax(1)
        total_loss += loss.item() * x.size(0)
        correct += (pred == y).sum().item()
        total += y.size(0)
        all_preds.extend(pred.cpu().tolist())
        all_true.extend(y.cpu().tolist())
    cm = confusion_matrix(all_preds, all_true, num_classes)
    return {
        "loss": total_loss / max(1, total),
        "acc": correct / max(1, total),
        "macro_f1": macro_f1_from_cm(cm),
        "cm": cm
    }

In [12]:
# =========================
# Warmup on labeled target only
# =========================
def run_labeled_epoch(loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(USE_AMP and device.type=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return {"loss": total_loss / max(1, total), "train_acc": correct / max(1, total)}

In [13]:
# =========================
# DANN + FixMatch epoch
# =========================
def get_cur_lambda_d(epoch_idx, total_epochs):
    if epoch_idx <= 3:
        return 0.0
    progress = (epoch_idx - 3) / max(1, total_epochs - 3)
    return LAMBDA_D_MAX * progress

def run_dann_fixmatch_epoch(source_loader, labeled_loader, unlabeled_loader, cur_lambda_d):
    model.train()
    total_loss = total_sup_loss = total_unsup_loss = total_dom_loss = 0.0
    total_correct = total_labeled = 0
    total_mask_selected = total_unlabeled = 0
    total_dom_correct = total_dom_count = 0

    source_iter = iter(source_loader)
    unlabeled_iter = iter(unlabeled_loader)

    for x_l, y_l in labeled_loader:
        try:
            x_s, _ = next(source_iter)
        except StopIteration:
            source_iter = iter(source_loader)
            x_s, _ = next(source_iter)

        try:
            x_u_w, x_u_s = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            x_u_w, x_u_s = next(unlabeled_iter)

        x_l, y_l = x_l.to(device), y_l.to(device)
        x_s = x_s.to(device)
        x_u_w = x_u_w.to(device)
        x_u_s = x_u_s.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(USE_AMP and device.type=="cuda")):
            logits_l = model(x_l)
            loss_sup = criterion(logits_l, y_l)

            with torch.no_grad():
                logits_u_w = model(x_u_w)
                probs_u_w = torch.softmax(logits_u_w, dim=1)
                max_probs, pseudo_labels = torch.max(probs_u_w, dim=1)
                mask = max_probs.ge(THRESHOLD).float()

            logits_u_s = model(x_u_s)
            per_sample_unsup = F.cross_entropy(logits_u_s, pseudo_labels, reduction="none")
            loss_unsup = (per_sample_unsup * mask).sum() / mask.sum().clamp(min=1.0)

            f_s = model.extract_features(x_s)
            f_t = model.extract_features(x_u_w)
            dom_logits_s = model.domain_logits(f_s, lambd=1.0)
            dom_logits_t = model.domain_logits(f_t, lambd=1.0)

            dom_labels_s = torch.zeros_like(dom_logits_s)
            dom_labels_t = torch.ones_like(dom_logits_t)
            loss_dom = dom_criterion(dom_logits_s, dom_labels_s) + dom_criterion(dom_logits_t, dom_labels_t)

            loss = loss_sup + LAMBDA_U * loss_unsup + cur_lambda_d * loss_dom

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x_l.size(0)
        total_sup_loss += loss_sup.item() * x_l.size(0)
        total_unsup_loss += loss_unsup.item() * x_l.size(0)
        total_dom_loss += loss_dom.item() * x_l.size(0)

        pred_l = logits_l.argmax(1)
        total_correct += (pred_l == y_l).sum().item()
        total_labeled += y_l.size(0)

        total_mask_selected += int(mask.sum().item())
        total_unlabeled += int(mask.numel())

        dom_pred_s = (torch.sigmoid(dom_logits_s) > 0.5).long()
        dom_pred_t = (torch.sigmoid(dom_logits_t) > 0.5).long()
        total_dom_correct += (dom_pred_s == 0).sum().item()
        total_dom_correct += (dom_pred_t == 1).sum().item()
        total_dom_count += dom_pred_s.numel() + dom_pred_t.numel()

    return {
        "loss": total_loss / max(1, total_labeled),
        "sup_loss": total_sup_loss / max(1, total_labeled),
        "unsup_loss": total_unsup_loss / max(1, total_labeled),
        "dom_loss": total_dom_loss / max(1, total_labeled),
        "train_acc": total_correct / max(1, total_labeled),
        "mask_rate": total_mask_selected / max(1, total_unlabeled),
        "domain_acc": total_dom_correct / max(1, total_dom_count),
    }

In [14]:
# =========================
# Train loop
# =========================
best_path = os.path.join(OUT_DIR, "dann_fixmatch_best.pt")
log_path = os.path.join(OUT_DIR, "dann_fixmatch_log.json")
cm_path = os.path.join(OUT_DIR, "dann_fixmatch_confusion_matrix.txt")

best_acc = -1.0
log = []

print("\n========== DANN + FixMatch (v2, correct loading) ==========")

for epoch in range(1, WARMUP_EPOCHS + 1):
    t0 = time.time()
    tr = run_labeled_epoch(pd_labeled_loader)
    ev = evaluate(pd_test_loader)

    row = {
        "stage": "warmup",
        "epoch": epoch,
        "train_loss": float(tr["loss"]),
        "train_acc": float(tr["train_acc"]),
        "pd_test_loss": float(ev["loss"]),
        "pd_test_acc": float(ev["acc"]),
        "pd_test_macro_f1": float(ev["macro_f1"]),
        "sec": float(time.time()-t0)
    }
    log.append(row)

    print(
        f"[Warmup] Epoch {epoch:03d}/{WARMUP_EPOCHS} | "
        f"train_labeled loss {tr['loss']:.4f} acc {tr['train_acc']:.4f} | "
        f"PlantDoc test loss {ev['loss']:.4f} acc {ev['acc']:.4f} macroF1 {ev['macro_f1']:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

    if ev["acc"] > best_acc:
        best_acc = ev["acc"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_acc": best_acc,
            "epoch": epoch,
            "stage": "warmup",
            "classes": source_classes,
            "model_name": MODEL_NAME,
            "start_ckpt": START_CKPT,
            "threshold": THRESHOLD,
            "lambda_u": LAMBDA_U,
            "lambda_d_max": LAMBDA_D_MAX,
        }, best_path)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    cur_lambda_d = get_cur_lambda_d(epoch, EPOCHS)
    tr = run_dann_fixmatch_epoch(src_loader, pd_labeled_loader, pd_unlabeled_loader, cur_lambda_d)
    ev = evaluate(pd_test_loader)

    row = {
        "stage": "dann_fixmatch",
        "epoch": epoch,
        "cur_lambda_d": float(cur_lambda_d),
        "train_loss": float(tr["loss"]),
        "sup_loss": float(tr["sup_loss"]),
        "unsup_loss": float(tr["unsup_loss"]),
        "dom_loss": float(tr["dom_loss"]),
        "train_acc": float(tr["train_acc"]),
        "mask_rate": float(tr["mask_rate"]),
        "domain_acc": float(tr["domain_acc"]),
        "pd_test_loss": float(ev["loss"]),
        "pd_test_acc": float(ev["acc"]),
        "pd_test_macro_f1": float(ev["macro_f1"]),
        "sec": float(time.time()-t0)
    }
    log.append(row)

    print(
        f"[DANN+Fix] Epoch {epoch:03d}/{EPOCHS} | "
        f"lambda_d {cur_lambda_d:.4f} | "
        f"train loss {tr['loss']:.4f} sup {tr['sup_loss']:.4f} unsup {tr['unsup_loss']:.4f} dom {tr['dom_loss']:.4f} "
        f"acc {tr['train_acc']:.4f} mask {tr['mask_rate']:.4f} domainAcc {tr['domain_acc']:.4f} | "
        f"PlantDoc test loss {ev['loss']:.4f} acc {ev['acc']:.4f} macroF1 {ev['macro_f1']:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

    if ev["acc"] > best_acc:
        best_acc = ev["acc"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_acc": best_acc,
            "epoch": epoch,
            "stage": "dann_fixmatch",
            "classes": source_classes,
            "model_name": MODEL_NAME,
            "start_ckpt": START_CKPT,
            "threshold": THRESHOLD,
            "lambda_u": LAMBDA_U,
            "lambda_d_max": LAMBDA_D_MAX,
        }, best_path)

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

final_ev = evaluate(pd_test_loader)
with open(cm_path, "w", encoding="utf-8") as f:
    f.write("Confusion Matrix (rows=true, cols=pred)\n")
    f.write(str(final_ev["cm"].tolist()) + "\n")
    f.write(f"Final Acc: {final_ev['acc']:.6f}\n")
    f.write(f"Final Macro-F1: {final_ev['macro_f1']:.6f}\n")

print("\n[Done] best PlantDoc test acc =", best_acc)
print("[Saved] best checkpoint:", best_path)
print("[Saved] training log:", log_path)
print("[Saved] confusion matrix:", cm_path)


========== DANN + FixMatch (v2, correct loading) ==========
[Warmup] Epoch 001/2 | train_labeled loss 0.5135 acc 0.8526 | PlantDoc test loss 2.1273 acc 0.4020 macroF1 0.3033 | 33.4s
[Warmup] Epoch 002/2 | train_labeled loss 0.3919 acc 0.9158 | PlantDoc test loss 2.1507 acc 0.4118 macroF1 0.3149 | 1.4s
[DANN+Fix] Epoch 001/20 | lambda_d 0.0000 | train loss 0.3900 sup 0.2810 unsup 0.4361 dom 1.4129 acc 0.9263 mask 0.2812 domainAcc 0.4648 | PlantDoc test loss 2.2286 acc 0.3725 macroF1 0.2784 | 40.3s
[DANN+Fix] Epoch 002/20 | lambda_d 0.0000 | train loss 0.2984 sup 0.2017 unsup 0.3868 dom 1.4047 acc 0.9684 mask 0.2344 domainAcc 0.4688 | PlantDoc test loss 2.2797 acc 0.3725 macroF1 0.2759 | 37.2s
[DANN+Fix] Epoch 003/20 | lambda_d 0.0000 | train loss 0.2899 sup 0.1761 unsup 0.4554 dom 1.3984 acc 0.9579 mask 0.3047 domainAcc 0.4922 | PlantDoc test loss 2.3884 acc 0.3627 macroF1 0.2645 | 26.9s
[DANN+Fix] Epoch 004/20 | lambda_d 0.0006 | train loss 0.2417 sup 0.1094 unsup 0.5257 dom 1.3957 ac